In [ ]:
using BayesSoundSource
using Plots 
using Distributions 
using Random
using Serialization
using Turing
using MCMCChains 
using ReverseDiff 

using ProgressMeter



In [ ]:

function plot_trajectory!(p, positions; c=missing)
    pos = Tuple.(positions)
    Plots.plot!(p, pos,c = c, alpha=0.3)
    scatter!(p, pos[1], c = c)
    scatter!(p, pos, marker=:xcross, c=c)
end


speed_of_sound = 343
σ_ToA = 0.0005
σ_TDoA = 0.0005

microphone_coords = [
[0.0, 0.0],
[-5.0, 1.5],
[5.0, 1.5],
[0.0, 3.0]
]

a = 7.0

p = Plots.plot(xlabel="X",ylabel="Y",title="2D Stochastic Trajectory",lw=2,legend=false)
Plots.scatter!(p, Tuple.(microphone_coords))  

microphone_uncertainty = MvNormal.(microphone_coords, 0.1)

paths = []


times = range(0, step=0.2, length=10)

i = 1
while(i <= 500)
    x0 = [rand(Uniform(-10, 10)), rand(Uniform(3, 8))]

    v0x = rand(Uniform(2, 5)) * -sign(x0[1])

    v0 = [v0x, rand(Uniform(-2, 2))]
    times, positions = stochastic_trajectory(a; d=2, tmax=2.0, Δt=0.2, x0=x0, v0=v0)#, seed=42)

    if any(getindex.(positions, 2) .<= 0) 
        continue
    end 
    i += 1

    tdoa = synthetic_tdoa(positions, microphone_coords, speed_of_sound, σ_TDoA)

    toa = synthetic_toa(positions, times, microphone_coords, speed_of_sound, σ_ToA)

    microphone_measurement = rand.(microphone_uncertainty)
    microphone_prior = MvNormal.(microphone_measurement, 0.1)

    push!(paths, (path_gt = positions, times_gt = times, tdoa = tdoa, toa = toa, xs_prior = microphone_prior))
    plot_trajectory!(p, positions, c=i)
end 

serialize(".data/paths.bin", paths)
p

In [ ]:
Turing.setprogress!(false);
n_samples = 1000

paths = deserialize(".data/paths.bin")

gp_prior = GPTrajPrior(2)
flat_prior = FlatTrajPrior(2, [(-30.0,30.0), (-30.0,30.0)])


@showprogress Threads.@threads for i in 1:length(paths)
    path = paths[i]
    
    if !(isfile(".data/chain_toa_$i.bin"))
        model = toa_model(path.toa, paths[1].xs_prior, flat_prior)
        
        chn = sample(model, 
                     NUTS(; adtype=AutoReverseDiff(compile=true)), 
                     n_samples; 
                     discard_adapt = true, 
                     verbose=false, 
                     initial_params=InitFromPrior(), 
                     chain_type=MCMCChains.Chains)
        serialize(".data/chain_toa_$i.bin", chn)
    end 


    if !(isfile(".data/chain_toa_gp_$i.bin"))
        model = toa_model(path.toa, paths[1].xs_prior, gp_prior)
        
        chn = sample(model, 
                     NUTS(; adtype=AutoReverseDiff(compile=true)), 
                     n_samples; 
                     discard_adapt = true, 
                     verbose=false, 
                     initial_params=InitFromPrior(), 
                     chain_type=MCMCChains.Chains)
        serialize(".data/chain_toa_gp_$i.bin", chn)
    end 


    if !(isfile(".data/chain_tdoa_$i.bin"))
        model = tdoa_model(path.tdoa, paths[1].xs_prior, flat_prior)
        
        chn = sample(model, 
                     NUTS(; adtype=AutoReverseDiff(compile=true)), 
                     n_samples; 
                     discard_adapt = true, 
                     verbose=false, 
                     initial_params=InitFromPrior(), 
                     chain_type=MCMCChains.Chains)
        serialize(".data/chain_tdoa_$i.bin", chn)
    end 


    if !(isfile(".data/chain_toa_tdoa_$i.bin"))
        model = toa_tdoa_model(path.tdoa, path.toa, paths[1].xs_prior, flat_prior)
        
        chn = sample(model, 
                     NUTS(; adtype=AutoReverseDiff(compile=true)), 
                     n_samples; 
                     discard_adapt = true, 
                     verbose=false, 
                     initial_params=InitFromPrior(), 
                     chain_type=MCMCChains.Chains)
        serialize(".data/chain_toa_tdoa_$i.bin", chn)
    end 

    if !(isfile(".data/chain_toa_tdoa_gp_$i.bin"))
        model = toa_tdoa_model(path.tdoa, path.toa, paths[1].xs_prior, gp_prior)
        
        chn = sample(model, 
                     NUTS(; adtype=AutoReverseDiff(compile=true)), 
                     n_samples; 
                     discard_adapt = true, 
                     verbose=false, 
                     initial_params=InitFromPrior(), 
                     chain_type=MCMCChains.Chains)
        serialize(".data/chain_toa_tdoa_gp_$i.bin", chn)
    end 
end 

In [ ]:
paths = deserialize(".data/paths.bin")

function point_error_map(pathmap, paths) 
    errors = []
    for i ∈ 1:length(paths)
        chn = deserialize(pathmap(i))
        path = paths[i]

        map_traj = map_estimate(chn, 2)
        for i ∈ 1:10
            push!(errors, distance(map_traj[i,:], path.path_gt[i]))
        end 
    end 
    return errors
end 


error_tdoa_LS = []
error_toa_LS = []
for i ∈ 1:length(paths)
    path = paths[i]
    tdoa_LS_est = map(path.tdoa) do tdoa_set 
        tdoa_mle_2d(mean.(path.xs_prior), tdoa_set, speed_of_sound)[1]
    end 
    for i ∈ 1:10
        push!(error_tdoa_LS, distance(tdoa_LS_est[i], path.path_gt[i]))
    end 

    toa_LS_est = map(path.toa) do toa_set 
        toa_mle_2d(mean.(path.xs_prior), toa_set, speed_of_sound)[1]
    end 
    for i ∈ 1:10
        push!(error_toa_LS, distance(toa_LS_est[i], path.path_gt[i]))
    end 
end 

errors_toa_tdoa_gp = point_error_map(i -> ".data/chain_toa_tdoa_gp_$i.bin", paths)
errors_toa_tdoa = point_error_map(i -> ".data/chain_toa_tdoa_$i.bin", paths)
errors_tdoa = point_error_map(i -> ".data/chain_tdoa_$i.bin", paths)
errors_toa_gp = point_error_map(i -> ".data/chain_toa_gp_$i.bin", paths)
errors_toa = point_error_map(i -> ".data/chain_toa_$i.bin", paths)

p = [0.05, 0.5, 0.95]
println("toa MLE       : ", round.(quantile(error_toa_LS, p), sigdigits=3))
println("tdoa MLE      : ", round.(quantile(error_tdoa_LS, p), sigdigits=3))
println("toa-tdoa + GP : ", round.(quantile(errors_toa_tdoa_gp, p), sigdigits=3))
println("toa-tdoa      : ", round.(quantile(errors_toa_tdoa, p), sigdigits=3))
println("tdoa          : ", round.(quantile(errors_tdoa, p), sigdigits=3))
println("toa      + GP : ", round.(quantile(errors_toa_gp, p), sigdigits=3))
println("toa           : ", round.(quantile(errors_toa, p), sigdigits=3))

In [ ]:
function spread_map(pathmap, paths) 
    errors = []
    for i ∈ 1:length(paths)
        chn = deserialize(pathmap(i))
        push!(errors, spread(chn, 2)...)

    end 
    return errors
end 
spread_toa_tdoa_gp = point_error_map(i -> ".data/chain_toa_tdoa_gp_$i.bin", paths)
spread_toa_tdoa = point_error_map(i -> ".data/chain_toa_tdoa_$i.bin", paths)
spread_tdoa = point_error_map(i -> ".data/chain_tdoa_$i.bin", paths)
spread_toa_gp = point_error_map(i -> ".data/chain_toa_gp_$i.bin", paths)
spread_toa = point_error_map(i -> ".data/chain_toa_$i.bin", paths)

p = [0.05, 0.5, 0.95]
println("toa-tdoa + GP : ", round.(quantile(spread_toa_tdoa_gp, p), sigdigits=3))
println("toa-tdoa      : ", round.(quantile(spread_toa_tdoa, p), sigdigits=3))
println("tdoa          : ", round.(quantile(spread_tdoa, p), sigdigits=3))
println("toa      + GP : ", round.(quantile(spread_toa_gp, p), sigdigits=3))
println("toa           : ", round.(quantile(spread_toa, p), sigdigits=3))

In [16]:
function sbc(pathmap, paths)
    N = length(paths)
    sbc_map = Vector{Float64}(undef, N*10)
    sbc_zero = Vector{Float64}(undef, N*10)
    sbc_time = Vector{Float64}(undef, N*10)
    idx = LinearIndices((N, 10))

    Threads.@threads for i in 1:N

        chn = deserialize(pathmap(i))
        path = paths[i]

        map_traj = map_estimate(chn, 2)
        traj = sample_traj(chn, 2)
        for j ∈ 1:10

            map_location = map_traj[j,:]

            location = traj[j,:,:]


            # Distance from MAP
            Df(sample) = distance(sample, map_location)
            R = Df.(eachcol(location))
            R_gt = Df(path.path_gt[j])

            sbc_map[idx[i,j]] = mean(R .<= R_gt)

            # Distance from ref
            Df_z(sample) = distance(sample, zeros(2))
            R = Df_z.(eachcol(location))
            R_gt = Df_z(path.path_gt[j])

            sbc_zero[idx[i,j]] = mean(R .<= R_gt)

            # Time difference 

            time = Array(group(chn, "t[$j]"))
            Df_t(sample) = sample 
            R = Df_t.(vec(time))
            R_gt = Df_t(path.times_gt[j])
            sbc_time[idx[i,j]] = mean(R .<= R_gt)

        end 
    end
    return sbc_map, sbc_zero, sbc_time
end 

sbcs_toa_tdoa_gp = sbc(i -> ".data/chain_toa_tdoa_gp_$i.bin", paths)
sbcs_toa_tdoa = sbc(i -> ".data/chain_toa_tdoa_$i.bin", paths)
sbcs_tdoa = sbc(i -> ".data/chain_tdoa_$i.bin", paths)
sbcs_toa_gp = sbc(i -> ".data/chain_toa_gp_$i.bin", paths)
sbcs_toa = sbc(i -> ".data/chain_toa_$i.bin", paths)

([0.178, 0.997, 0.961, 0.915, 0.763, 0.993, 0.149, 0.982, 0.955, 0.912  …  0.926, 0.76, 0.578, 0.854, 0.991, 0.498, 0.837, 0.054, 0.795, 0.929], [0.121, 0.003, 0.029, 0.016, 0.085, 0.007, 0.204, 0.008, 0.045, 0.087  …  0.045, 0.01, 0.189, 0.06, 0.008, 0.093, 0.038, 0.356, 0.003, 0.037], [0.869, 0.997, 0.981, 0.981, 0.9, 0.993, 0.758, 0.994, 0.959, 0.921  …  0.961, 0.981, 0.806, 0.937, 0.994, 0.864, 0.969, 0.633, 0.998, 0.962])

In [ ]:
using LaTeXStrings
using CairoMakie

U = Binomial(length(paths) * 10, 1/20)

μ = mean(U)
lo = quantile(U, 0.005)
hi = cquantile(U, 0.005)

fig = Figure(size=(1000, 600))

axes_map = [Axis(fig[2, i]) for i in 1:5]
axes_zero = [Axis(fig[3, i]) for i in 1:5]
axes_time = [Axis(fig[4, i]) for i in 1:5]

palette = Makie.wong_colors() 

# hide redundant y-axis decorations
for (i, ax) in enumerate(vcat(axes_map..., axes_zero..., axes_time...))
    ax.yticksvisible = false
    ax.yticklabelsvisible = false
    ax.xticksvisible = false 
    ax.xticklabelsvisible = false
    ax.limits = (0,1,0,μ*3)
end
Label(fig[1, 1], "ToA", tellwidth = false)
Label(fig[1, 2], "ToA + GP", tellwidth = false)
Label(fig[1, 3], "TDoA",  tellwidth = false)  
Label(fig[1, 4], "ToA-TDoA", tellwidth = false)
Label(fig[1, 5], "ToA-TDoA + GP", tellwidth = false)
rowsize!(fig.layout, 1, Fixed(1))
Label(fig[5, 1:5], "Rank Statistic", tellwidth = false)

axes_map[1].ylabel = L"h_{\text{MAP}}"
axes_zero[1].ylabel = L"h_{\text{ref}}"
axes_time[1].ylabel = L"h_{\text{time}}"

colgap!(fig.layout, 7)

linkaxes!(axes_map...)
linkaxes!(axes_zero...)
linkaxes!(axes_time...)

for i ∈ 1:3 
      datasets = [sbcs_toa[i], sbcs_toa_gp[i], sbcs_tdoa[i], sbcs_toa_tdoa[i], sbcs_toa_tdoa_gp[i]]
      axes = [axes_map, axes_zero, axes_time][i]

      for (ax, x) in zip(axes, datasets)
            if !all(isnan, x)
                  hist!(
                        ax,
                        x;
                        bins = range(0, 1, 20), 
                        color = palette[1]
                  )
                  # Gray confidence band
                  band!( ax, [0, 1], [lo, lo], [hi, hi], color = (:gray, 0.2))
                  hlines!( ax, [μ], color = :gray, linewidth = 2, label = "Uniform (0.95 CI)" )
            else 

                  hidedecorations!(ax)

                  lines!(ax, [0, 1], [0, 1],
                  space = :relative,
                  color = :gray,
                  linewidth = 4
                  )

                  lines!(ax, [0, 1], [1, 0],
                  space = :relative,
                  color = :gray,
                  linewidth = 4
                  )
            end
      end
end 
fig